# 007 · Dilution model

**The question.** Will this company issue many new shares soon?

**The answer.** `dilution`: 1 if shares outstanding (split-adjusted, as first filed) rise by 10% or more within 182 days, else 0. About 10% of rows are 1.

**The baseline to beat.** the base rate: the same chance (9.59%) for every row; its Brier score on all learnable rows is 0.0867 (the earlier 0.0898 predates V2-27).

**Measures.** Log loss (primary), AUC, Brier score and lift in the top tenth.

**Status.** First complete version (1 November 2026, branch `007-dilution-model`, spec in
`specs/007-dilution-model/`): four levels compared, gradient boosting chosen, frozen candidate
saved; waiting for the researcher's approval. Level 3 uses scikit-learn's
`HistGradientBoostingClassifier` as a stand-in for LightGBM (research R10, a D60 deviation to
confirm).

### What the answer means (read this first)

Think of an exam: the 96 **features** are the questions (what was known about a company on a
month-end: cash, losses, recent filings, price...), and **`dilution`** is the answer key (what
happened next). The model learns from rows *with* the key and is judged on rows whose key it never
saw.

- **The answer:** 1 if the company's shares outstanding (as filed on its 10-Q and 10-K covers,
  split-adjusted) rose **10% or more within the next 182 days** (6 months), 0 if not, missing if
  no new count was filed. Example: Company X on 31 March 2021 had 50 million shares (last count
  filed by then); the last count filed by 29 September 2021 says 58 million: +16%, so 1. With 52
  million (+4%), 0. With no new filing, missing (the row is not used).
- **Why 10%, not "any rise":** share counts creep up almost everywhere (employee stock: the median
  company +0.6% a year; two thirds of rows rise at all), so "any rise" would flag most healthy
  companies. What hurts shareholders is a share sale, which shows as a jump; above 10% the typical
  jump is +27%. The 10% is pre-registered (`config/risk_params.yaml` v5) and stays (decided 30
  Oct; an "offering event" question is roadmap 024).
- **Filings vs share counts:** filings (offering prospectuses, shelf registrations, 8-K 3.02
  unregistered sales) come early and are **features**: warning signs (after an offering filed in
  the past year, 21% of rows dilute 10%+ next; after an 8-K unregistered sale 27%; with none of
  these warning signs 2.3%). The share count is
  late but true, so it is the **answer**. The app shows the chance with these warning signs beside
  it.
- **Which rows and columns:** fit only on `dilution.parquet`, rows with `eligible_dilution` true
  and `dilution` present (409,576; base rate 9.6%); columns: the 96 features, never `dilution`
  or `window_end`. Save the feature list with the model.
- **How error is measured, in three stages, always on answers the model has not seen:** (1) the
  walk-forward test years 2019–2022 inside `dilution.parquet` (train on earlier years, score the
  year: this chooses the model); (2) the holdout, once (feature 010): the model predicts the
  2023+ month-end rows of `feature_snapshot.parquet`, which have **features but no answers** on
  purpose; the guarded command then computes their real answers from the share counts filed since
  and scores the predictions; (3) live (feature 009): each day's forecast is stored and scored
  six months later. `latest.parquet` is each company's newest row, for today's forecast.

### Rules for every model notebook

- **Spec first.** Before filling this notebook, run `/speckit-specify` for this feature with
  `SPECIFY_FEATURE_DIRECTORY=specs/007-dilution-model` on branch `007-dilution-model` (README, "Working
  together").
- **This notebook is where the model is built.** Every level is created, run and tuned here,
  with its tables and charts, start to finish by one command, so the researcher can read how the
  choice was made. Once the researcher is happy with the chosen model, its code moves into
  `src/sentinel/models/` (training, scoring, calibration) so that `sentinel update` scores every
  company each night without a notebook and the repository runs from a clone; this notebook then
  imports that code and must give the same numbers (docs/plan.md, "Notebooks and code").
- **Data.** Read only `data/exports/ml/dilution.parquet` through `sentinel.eda.load("dilution")`, which keeps
  the development rows (dated 2022 or earlier). Learn and test only on rows where this question's
  flag `eligible_dilution` is true and the answer is present (V2-26, V2-27). Never read `data/exports/reference/` (answers for 2023 on) and
  never compute anything for 2023 or later: that is the sealed holdout, tested once by feature 010
  (constitution IV).
- **Never use as features:** `dilution`, `window_end`, `cik`, `as_of` and the flags.
- **Never a random train/test split.** Rows of one company over time are not independent, and a
  random split lets the model learn from the future. The only split is walk-forward by year:
- **Evaluation is fixed** (decision D60, `config/risk_params.yaml` v5, never edited): yearly
  walk-forward test years 2019, 2020, 2021, 2022; for each, train only on rows whose answer window
  ended before 1 January of that year (`sentinel.eda.purge_table` counts them); four levels:
  naive baseline → rule → classic model → LightGBM; choose the **simplest level within one
  standard error of the best**; report every test year, not only the average.
- **Show forecasts as a chance in five bands, each with its real dilution rate on the test years** with their hit rates on the test years (D62), never as a bare
  number.
- **Reproducible:** sort rows by `cik`, `as_of` before modelling; fixed seeds (`SEED = 0`); log
  each run in MLflow with the git commit and the export date. Two runs must give identical
  predictions.
- **Private data:** outputs are stripped on commit (nbstripout); written conclusions quote
  aggregates only, never one company's prices or returns (Tiingo terms). Never write to `data/`.
- **Run it** start to finish under the memory cap:
  `systemd-run --user --scope -p MemoryMax=5G -p MemorySwapMax=0 uv run jupyter nbconvert --to notebook --execute --inplace notebooks/007-dilution-model.ipynb`

### Two kinds of test, and the final one is not in this notebook

1. **Test years 2019–2022 (here).** For each test year, train on the earlier years only (with
   the purge) and score that year. This is where the four levels are compared and **one is
   chosen**. Because the choice is made by looking at these results, they flatter the winner a
   little.
2. **The holdout: answers whose windows start in 2023 or later (not here, not in any notebook).**
   It answers the only question that matters in the end: does the model chosen here work on data
   nobody looked at while choosing it? That works only if it is used **once**.

**How it stays hidden.** The training file has no answers after 2022: the pipeline does not even
compute them (the label code stops at windows ending 31 December 2022). This notebook must not
read the private reference copies in `data/exports/reference/` (feature 002), which hold them.

**How the final test is run (feature 010, 16 November).**

1. This feature ends with the chosen model **frozen** in `src/sentinel/models/`: its settings,
   its training rows (to 2022), its seed, the git commit and the export date, logged in MLflow.
   The report (008) is written from the 2019–2022 results.
2. The researcher then runs **one guarded command** (built in 010). It checks that the models are
   frozen and that no holdout run exists yet, computes the 2023+ answers, scores the frozen
   models, and writes the results with a log entry. A second run is refused: the log is the proof
   that the test happened once.
3. The results go into the report as they came out. The model is not changed afterwards
   (constitution IV). Later refits (021) may use the 2023+ answers, as a logged step.

A command, not a notebook, because a notebook can be re-run with one click and nothing records
how often it was looked at.

### Make the model holdout-ready (so the one test runs without a hitch)

The holdout rows come from the same feature table as the training rows (`feature_snapshot`
from 2023 on: same columns, same order, same types, checked on 30 October) and their answers from
the same label code with a later end date. What can still break is the model itself, so:

1. **Select features by name, never by position**: save the list of feature columns with the
   model and use `rows[features]` when scoring.
2. **Every preprocessing step lives inside the model** (cut points, fill values, missing flags,
   scaling), fitted on the training rows only and saved with it, never recomputed on new rows.
3. **Save one object** (in MLflow): the fitted preprocessing and model, the feature list, the
   eligibility flag it uses (`{elig}`), the seed, the git commit and the export date.
4. **Dry run before 16 November**: load the saved model and score `latest.parquet` and the 2023+
   rows of `feature_snapshot.parquet` (features only, no answers): it must run, give one
   prediction per eligible row, no unexpected missing values, values in a plausible range. This
   looks at no outcome, so it does not spend the holdout.
5. **Scoring is a function** `predict(model, rows) -> predictions` in `src/sentinel/models/`,
   the same one used for the test years, the dry run, the holdout and the daily scores.

### What the EDA found for this model (notebooks/004-eda-dilution.ipynb, Findings)

- **Rule level ready-made:** "an offering filed in the last 12 months" (`n_offering_12m > 0`):
  AUC 0.70–0.73 in every test year; dilution 21.8% if flagged, 3.8% if not. A model must clearly
  beat it.
- Strongest features: offerings and shelf filings, share growth over 12 months, interest
  coverage (low or missing), R&D intensity, stock-based pay to revenue, volatility, low price,
  small size, loss-making. Macro and short volume add nothing (drop them from the classic
  model).
- Missing values carry meaning (no interest coverage often means a small company without debt):
  add "was missing" flags.
- **Foreign filers are left out of this question (V2-27, decided 30 Oct):** train, test and
  score only rows with `foreign_filer == False`; 78% of their rows have no answer and the rest
  are not typical.
- Rows with a stock split in the window dilute 25% vs 9.7%: open question V2-25 (named cases
  first). Impossible share counts (V2-23) can make wrong answers.

**Data cleaning** (V2-33): the pipeline's cleaning is frozen at V2-32 (foreign market values,
share-count typos, insider value typos, placeholder prices, splits in the dilution answer, Tiingo
instruments; `docs/decisions.md`). Any further cleaning is part of **this model**: cut extreme
values at the 1st and 99th percentiles of the training rows, drop rows with a known doubt, add
"was missing" flags; write it here, explain why, and move it into `src/sentinel/models/` with the
model so the daily scoring cleans the same way. Do not rebuild the database for small fixes.

## 1. Setup and data

*To do:* Load the development rows, keep eligible rows with an answer, list the features to use (the EDA's advice below), and print rows and companies per test year.

In [ ]:
# Setup: libraries, the rows the model learns from, the features, MLflow
import hashlib
import subprocess
import tempfile
import warnings
from dataclasses import dataclass
from datetime import date
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

from sentinel import config, eda

SEED = 0
THREADS = 4  # keep the Mac usable while it trains
threadpool_limits(THREADS)
warnings.filterwarnings("ignore", category=RuntimeWarning)  # nan-quantiles of empty columns
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

ANSWER, ELIG = "dilution", "eligible_dilution"
raw = eda.load("dilution")
rows = (raw[raw[ELIG] & raw[ANSWER].notna()]
        .sort_values(["cik", "as_of"], kind="stable").reset_index(drop=True))
y = rows[ANSWER].astype(int)
assert len(rows) == 409_576 and round(y.mean() * 100, 2) == 9.59, (len(rows), y.mean())

# Features: the export's 96, minus cum_split_after (product of split ratios AFTER the row's day:
# look-ahead, research R1). Identifiers, flags, the answer and its window are never features.
LOOK_AHEAD = ["cum_split_after"]
FEATURES = [c for c in eda.features(rows) if c not in LOOK_AHEAD]
NEVER = {"dilution", "window_end", "cik", "as_of", "cum_split_after", "month_end", "release_eve",
         "latest", "coverage", "filer_type", "foreign_filer", "sector", "active_filer",
         "excluded", "eligible", "eligible_volatility", "eligible_dilution"}
assert not NEVER & set(FEATURES) and len(FEATURES) == 95
FAMILY = {c: eda.family(c) for c in FEATURES}
MACRO = [c for c in FEATURES if FAMILY[c] == "macro"]
SHORT = [c for c in FEATURES if FAMILY[c] == "short volume"]
INSIDER = [c for c in FEATURES if FAMILY[c] == "insider"]
X_ALL = rows[FEATURES].astype(float)

print(f"{len(rows):,} rows, {rows.cik.nunique():,} companies, base rate {y.mean():.2%}")
print(f"{len(FEATURES)} features (left out: {LOOK_AHEAD})")
display(rows.assign(year=rows.as_of.dt.year).groupby("year")
        .agg(rows=("cik", "size"), companies=("cik", "nunique"), dilution_rate=(ANSWER, "mean")))

# MLflow: runs are logged to data/mlflow.db, artifacts to data/mlartifacts (research R9)
P = config.paths()
mlflow.set_tracking_uri(f"sqlite:///{P.mlflow_db}")
EXPERIMENT = "007-dilution"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT,
                             artifact_location=(P.mlflow_db.parent / "mlartifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
SRC = eda.EXPORT / "dilution.parquet"
TAGS = {
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                                 text=True).stdout.strip(),
    "export_date": date.fromtimestamp(SRC.stat().st_mtime).isoformat(),
    "data_hash": hashlib.sha256(SRC.read_bytes()).hexdigest()[:12],
    "level3_estimator": "sklearn HistGradientBoostingClassifier (stand-in for LightGBM, R10)",
}
print(TAGS)

## 2. Walk-forward folds

*To do:* Build the four folds (test years 2019–2022) with the purge; print train and test rows per fold.

In [ ]:
# Walk-forward folds with the purge, and the inner validation year (research R3)
def fold(year: int) -> dict:
    """Row masks for test year `year`: train (window ended before 1 Jan), inner-train (window
    ended before the last training year), valid (the last training year), test (the year)."""
    start, valid_start = pd.Timestamp(year, 1, 1), pd.Timestamp(year - 1, 1, 1)
    train = rows.window_end < start
    return {"train": train, "inner": rows.window_end < valid_start,
            "valid": train & (rows.as_of >= valid_start), "test": rows.as_of.dt.year == year,
            "purged": (rows.as_of < start) & (rows.window_end >= start)}


FOLDS = {year: fold(year) for year in eda.TEST_YEARS}
fold_table = pd.DataFrame({year: {k: int(m.sum()) for k, m in f.items()}
                           for year, f in FOLDS.items()}).T
display(fold_table)
expected = eda.purge_table(rows, "window_end")
assert (fold_table[["train", "purged", "test"]].values == expected[
    ["train_rows", "purged", "test_rows"]].values).all()
print("fold counts equal the EDA's purge table")


def scores(y_true, p) -> dict:
    """Log loss (primary), AUC, Brier score and top-tenth lift of chances `p`."""
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    yt, ps = pd.Series(np.asarray(y_true, dtype=float)), pd.Series(p)
    return {"log_loss": log_loss(yt, p, labels=[0, 1]), "auc": eda.auc(yt, ps),
            "brier": brier_score_loss(yt, p), "lift_top10": eda.top_decile_lift(yt, ps)}


RESULTS, TEST_PRED, VALID_PRED, FITTED = [], {}, {}, {}


def record(level: str, year: int, p_test, params: dict, p_valid=None, extra=None):
    """Keep a level's test-year chances and scores, and log them as one MLflow run."""
    m = scores(y[FOLDS[year]["test"]], p_test)
    RESULTS.append({"level": level, "year": year, **m})
    TEST_PRED[(level, year)] = np.asarray(p_test, dtype=float)
    if p_valid is not None:
        VALID_PRED[(level, year)] = np.asarray(p_valid, dtype=float)
    with mlflow.start_run(run_name=f"{level}-{year}"):
        mlflow.set_tags({**TAGS, "level": level, "test_year": year})
        mlflow.log_params(params)
        mlflow.log_metrics({**m, **(extra or {})})
    return m


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def platt(p_valid, y_valid) -> LogisticRegression:
    """Platt scaling: a logistic fit of the answer on the model's log-odds (research R6)."""
    return LogisticRegression(C=1e6).fit(logit(p_valid)[:, None], y_valid)


def calibrate(cal, p):
    return cal.predict_proba(logit(p)[:, None])[:, 1]

## 3. Level 0: the naive baseline

*To do:* Score the baseline on each test year with the chosen measures.

In [ ]:
# Level 0: the base rate of each fold's training rows, the same chance for every row
for year, f in FOLDS.items():
    rate = y[f["train"]].mean()
    record("base_rate", year, np.full(int(f["test"].sum()), rate), {"rate": round(rate, 6)})
display(pd.DataFrame(RESULTS).query("level == 'base_rate'").set_index("year"))
print("Brier score of always saying the base rate is about 0.0867: the number to beat")

## 4. Level 1: a rule

*To do:* A simple, explainable rule (see the EDA's suggestion below); fit only on each fold's training rows.

In [ ]:
# Level 1: the EDA's rule, "an offering was filed in the last 12 months" (research R4)
FLAG = (rows.n_offering_12m > 0).to_numpy()
for year, f in FOLDS.items():
    tr = f["train"].to_numpy()
    rates = {"flagged": y[tr & FLAG].mean(), "not": y[tr & ~FLAG].mean()}
    p = np.where(FLAG[f["test"].to_numpy()], rates["flagged"], rates["not"])
    m = record("rule", year, p, {k: round(v, 6) for k, v in rates.items()})
    assert 0.69 <= m["auc"] <= 0.76, m
display(pd.DataFrame(RESULTS).query("level == 'rule'").set_index("year"))

# For context only (not levels): the other simple rules' AUC per test year
context = {"offering filed in 12 months": FLAG,
           "shelf effective in 12 months": (rows.n_shelf_effective_12m > 0).to_numpy(),
           "shares up 10%+ in 12 months": (rows.share_growth_12m >= 0.10).to_numpy()}
display(pd.DataFrame({year: {name: eda.auc(y[f["test"]], pd.Series(flag, index=rows.index)[f["test"]]
                                           .astype(float))
                             for name, flag in context.items()}
                      for year, f in FOLDS.items()}))

## 5. Level 2: a classic model

*To do:* A linear or logistic model: one feature of each redundant pair, ratios cut at their 1st and 99th percentiles (on training rows), missing values filled with the training median plus a 'was missing' flag, standardised.

In [ ]:
# Level 2: a regularised logistic regression with the model-side cleaning of V2-33
class Clip(BaseEstimator, TransformerMixin):
    """Cut each column at its 1st and 99th percentiles of the rows it was fitted on."""

    def __init__(self, lower=0.01, upper=0.99):
        self.lower, self.upper = lower, upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lo_ = np.nanquantile(X, self.lower, axis=0)
        self.hi_ = np.nanquantile(X, self.upper, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)


# Columns: no macro, no short volume; one of each |Spearman| >= 0.9 pair, chosen on the 2019
# fold's training rows only (fixed for all folds), keeping the log versions.
KEEP = {"log_addv20", "log_market_cap", "log_close"}
candidates = [c for c in FEATURES if c not in MACRO + SHORT]
pairs = eda.redundant_pairs(rows[FOLDS[2019]["train"]], candidates, threshold=0.9, seed=SEED)
dropped = []
for a, b in zip(pairs.a, pairs.b):
    if a in dropped or b in dropped or (a in KEEP and b in KEEP):
        continue
    dropped.append(a if b in KEEP else b)
LINEAR = [c for c in candidates if c not in dropped]
print(f"{len(LINEAR)} columns; left out as redundant: {dropped}")


def logistic_pipeline(C: float):
    return make_pipeline(Clip(), SimpleImputer(strategy="median", add_indicator=True),
                         StandardScaler(), LogisticRegression(C=C, max_iter=3000))


def fit_logistic(f: dict) -> dict:
    """Choose C on the validation year, fit on inner-train rows, calibrate on validation."""
    Xi, yi = X_ALL.loc[f["inner"], LINEAR].to_numpy(), y[f["inner"]].to_numpy()
    Xv, yv = X_ALL.loc[f["valid"], LINEAR].to_numpy(), y[f["valid"]].to_numpy()
    tried = {}
    for C in (0.01, 0.1, 1.0):
        pipe = logistic_pipeline(C).fit(Xi, yi)
        tried[C] = (log_loss(yv, pipe.predict_proba(Xv)[:, 1]), pipe)
    C = min(tried, key=lambda c: tried[c][0])
    pipe = tried[C][1]
    p_valid = pipe.predict_proba(Xv)[:, 1]
    return {"estimator": pipe, "calibrator": platt(p_valid, yv), "params": {"C": C},
            "features": LINEAR, "valid_raw": p_valid,
            "tried": {c: round(v[0], 5) for c, v in tried.items()}}


for year, f in FOLDS.items():
    fit = fit_logistic(f)
    FITTED[("logistic", year)] = fit
    raw_test = fit["estimator"].predict_proba(X_ALL.loc[f["test"], LINEAR].to_numpy())[:, 1]
    p_valid = calibrate(fit["calibrator"], fit["valid_raw"])
    record("logistic", year, calibrate(fit["calibrator"], raw_test), fit["params"], p_valid,
           {"log_loss_raw": log_loss(y[f["test"]], raw_test)})
    print(year, "C tried (validation log loss):", fit["tried"], "-> C =", fit["params"]["C"])
display(pd.DataFrame(RESULTS).query("level == 'logistic'").set_index("year"))

## 6. Level 3: LightGBM

*To do:* Gradient-boosted trees with fixed seeds and settings chosen by time-ordered validation inside the training years only (never on the test year).

In [ ]:
# Level 3: gradient-boosted trees. D60 names LightGBM; on this Mac scikit-learn's
# HistGradientBoostingClassifier stands in (same histogram boosting, research R10). Switching back
# to LightGBM means changing only `boosting_model` and the grid names below.
GRID = [{"max_leaf_nodes": n, "min_samples_leaf": m} for n in (15, 63) for m in (100, 500)]


def boosting_model(params: dict, seed: int = SEED, early_stopping: bool = True, max_iter=2000):
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=max_iter, max_features=0.8, random_state=seed,
        early_stopping=early_stopping, n_iter_no_change=100, scoring="loss", **params)


def fit_boosting(f: dict, features=None, seed: int = SEED, params=None) -> dict:
    """Choose settings on the validation year (time-ordered early stopping), fit on inner-train
    rows, calibrate on validation. With `params` given, only that setting is fitted."""
    features = features or FEATURES
    Xi, yi = X_ALL.loc[f["inner"], features].to_numpy(), y[f["inner"]].to_numpy()
    Xv, yv = X_ALL.loc[f["valid"], features].to_numpy(), y[f["valid"]].to_numpy()
    tried = {}
    for p in ([params] if params else GRID):
        model = boosting_model(p, seed).fit(Xi, yi, X_val=Xv, y_val=yv)
        tried[tuple(p.items())] = (log_loss(yv, model.predict_proba(Xv)[:, 1]), model)
    best = min(tried, key=lambda k: tried[k][0])
    model = tried[best][1]
    p_valid = model.predict_proba(Xv)[:, 1]
    return {"estimator": model, "calibrator": platt(p_valid, yv),
            "params": {**dict(best), "n_iter": int(model.n_iter_)}, "features": features,
            "valid_raw": p_valid, "tried": {str(dict(k)): round(v[0], 5) for k, v in tried.items()}}


for year, f in FOLDS.items():
    fit = fit_boosting(f)
    FITTED[("boosting", year)] = fit
    raw_test = fit["estimator"].predict_proba(X_ALL.loc[f["test"], FEATURES].to_numpy())[:, 1]
    p_valid = calibrate(fit["calibrator"], fit["valid_raw"])
    record("boosting", year, calibrate(fit["calibrator"], raw_test), fit["params"], p_valid,
           {"log_loss_raw": log_loss(y[f["test"]], raw_test)})
    print(year, "settings tried (validation log loss):", fit["tried"], "->", fit["params"])
display(pd.DataFrame(RESULTS).query("level == 'boosting'").set_index("year"))

## 7. Comparison and choice

*To do:* One table: each level × each test year, the mean and its standard error; pick the simplest level within one standard error of the best.

In [ ]:
# Comparison of the four levels and the one-standard-error choice (research R5)
ORDER = ["base_rate", "rule", "logistic", "boosting"]  # simplest first
res = pd.DataFrame(RESULTS)
for measure in ["log_loss", "auc", "brier", "lift_top10"]:
    print(measure)
    display(res.pivot(index="level", columns="year", values=measure).loc[ORDER])

summary = res.groupby("level")["log_loss"].agg(mean="mean", sd="std").loc[ORDER]
summary["se"] = summary["sd"] / np.sqrt(len(FOLDS))
best = summary["mean"].idxmin()
threshold = summary.loc[best, "mean"] + summary.loc[best, "se"]
CHOSEN = next(level for level in ORDER if summary.loc[level, "mean"] <= threshold)
summary["within_one_se_of_best"] = summary["mean"] <= threshold
display(summary)
print(f"best mean log loss: {best} ({summary.loc[best, 'mean']:.5f}); "
      f"one SE above it: {threshold:.5f}")
print(f"CHOSEN LEVEL: {CHOSEN} (the simplest level within one standard error of the best)")

wide = res.pivot(index="year", columns="level", values=["log_loss", "auc"])
beats_rule = ((wide["log_loss"][CHOSEN] < wide["log_loss"]["rule"])
              & (wide["auc"][CHOSEN] > wide["auc"]["rule"]))
beats_base = res.query("level == @CHOSEN").set_index("year")["brier"] < res.query(
    "level == 'base_rate'").set_index("year")["brier"]
display(pd.DataFrame({"beats rule (log loss and AUC)": beats_rule,
                      "beats base-rate Brier": beats_base}))
print("Note: the 2022 test year covers January-June only (later windows end after 2022).")

## 8. Calibration and bands

*To do:* Calibrate the chances on earlier years (for example isotonic or Platt on the training years' last part); five bands by predicted chance, each with its real dilution rate per test year (D47).

In [ ]:
# Five bands with fixed edges from the validation year, and calibration (research R6)
BAND_LEVEL = CHOSEN if CHOSEN in ("logistic", "boosting") else "boosting"
if BAND_LEVEL != CHOSEN:
    print(f"The chosen level ({CHOSEN}) gives at most two chances, so bands are shown for "
          f"{BAND_LEVEL} for comparison.")
bands, monotone = [], {}
for year, f in FOLDS.items():
    edges = np.quantile(VALID_PRED[(BAND_LEVEL, year)], [0.2, 0.4, 0.6, 0.8])
    p = TEST_PRED[(BAND_LEVEL, year)]
    band = np.searchsorted(edges, p, side="right") + 1
    t = (pd.DataFrame({"band": band, "chance": p, "real": y[f["test"]].to_numpy()})
         .groupby("band").agg(rows=("real", "size"), forecast=("chance", "mean"),
                              real_rate=("real", "mean")))
    monotone[year] = bool(t["real_rate"].is_monotonic_increasing)
    bands.append(t.assign(year=year).reset_index())
band_table = pd.concat(bands).pivot(index="band", columns="year",
                                    values=["rows", "forecast", "real_rate"])
display(band_table)
print("real rate rises from band 1 to band 5:", monotone)

fig, axes = plt.subplots(1, len(FOLDS), figsize=(16, 4), sharey=True)
for ax, (year, f) in zip(axes, FOLDS.items()):
    d = pd.DataFrame({"p": TEST_PRED[(BAND_LEVEL, year)], "y": y[f["test"]].to_numpy()})
    d["group"] = pd.qcut(d["p"].rank(method="first"), 10, labels=False)
    g = d.groupby("group").mean()
    ax.plot([0, g.max().max()], [0, g.max().max()], color="grey", lw=1, ls="--")
    ax.plot(g["p"], g["y"], marker="o")
    ax.set_title(f"{year}")
    ax.set_xlabel("forecast chance")
axes[0].set_ylabel("real dilution rate")
fig.suptitle(f"Calibration of {BAND_LEVEL}, ten groups per test year")
plt.show()

## 9. What the model uses

*To do:* Feature importance (LightGBM) or coefficients (classic); check that the strongest features make sense and none is a leak.

In [ ]:
# What the models use, a leak check, and the insider/macro family check (research R7)
coef = FITTED[("logistic", 2021)]["estimator"]
lr = coef[-1]
imputer = coef[1]
labels = LINEAR + [f"{LINEAR[i]} (was missing)" for i in imputer.indicator_.features_]
top_coef = (pd.Series(lr.coef_[0], index=labels).sort_values(key=np.abs, ascending=False)
            .head(20).rename("standardised coefficient (2021 fold)"))
display(top_coef.to_frame())

f21 = FOLDS[2021]
boost = FITTED[("boosting", 2021)]["estimator"]
sample = rows.index[f21["test"]].to_series().sample(20_000, random_state=SEED)
perm = permutation_importance(boost, X_ALL.loc[sample, FEATURES].to_numpy(), y[sample].to_numpy(),
                              scoring="neg_log_loss", n_repeats=3, random_state=SEED)
top_perm = (pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
            .head(20).rename("log-loss increase when shuffled (2021 test sample)"))
display(top_perm.to_frame().assign(family=lambda t: [FAMILY[c] for c in t.index]))

# Leak check: inputs are features only, and no single feature is near-perfect on training rows
assert set(LINEAR) <= set(FEATURES) and not NEVER & set(LINEAR + FEATURES)
tr19 = FOLDS[2019]["train"]
single = pd.Series({c: eda.auc(y[tr19], X_ALL.loc[tr19, c]) for c in FEATURES
                    if X_ALL.loc[tr19, c].notna().sum() > 1000})
suspicious = single[(single >= 0.95) | (single <= 0.05)]
print("features with single AUC >= 0.95 or <= 0.05 on 2019 training rows:", list(suspicious.index))
assert suspicious.empty

# Family check (D60): refit the trees without insider, and without macro, seeds 0-4, every fold,
# with each fold's chosen settings; change in test log loss against the full model.
family_rows = []
for year, f in FOLDS.items():
    params = {k: v for k, v in FITTED[("boosting", year)]["params"].items() if k != "n_iter"}
    for name, cols in {"full": FEATURES,
                       "without insider": [c for c in FEATURES if c not in INSIDER],
                       "without macro": [c for c in FEATURES if c not in MACRO]}.items():
        for seed in range(5):
            fit = fit_boosting(f, cols, seed, params)
            raw_test = fit["estimator"].predict_proba(X_ALL.loc[f["test"], cols].to_numpy())[:, 1]
            p = calibrate(fit["calibrator"], raw_test)
            family_rows.append({"year": year, "model": name, "seed": seed,
                                "log_loss": log_loss(y[f["test"]], p)})
fam = pd.DataFrame(family_rows)
full = fam.query("model == 'full'").groupby(["year", "seed"])["log_loss"]
fam["change"] = fam["log_loss"] - fam.set_index(["year", "seed"]).index.map(full.first())
family_table = (fam.query("model != 'full'").groupby(["model", "year"])["change"]
                .agg(mean="mean", sd="std").unstack("year"))
display(family_table)
print("positive change = worse without the family, so the family helps")
with mlflow.start_run(run_name="family-check"):
    mlflow.set_tags({**TAGS, "level": "boosting", "check": "family"})
    for (model, year), v in fam.query("model != 'full'").groupby(["model", "year"])["change"]:
        mlflow.log_metric(f"{model.replace(' ', '_')}_{year}_change", v.mean())

## 10. Frozen model, dry run and findings

The chosen level refitted for the holdout year (2023) as one object with a `predict` function (contracts/frozen-model.md), a dry run on rows without answers, then the findings.

In [ ]:
# The frozen model: one object and one predict function (contracts/frozen-model.md)
@dataclass
class DilutionModel:
    level: str
    features: list
    eligibility: str
    estimator: object
    calibrator: object
    band_edges: list
    seed: int
    git_commit: str
    export_date: str
    data_hash: str
    trained_through: str


def _raw_chance(model: DilutionModel, X: pd.DataFrame) -> np.ndarray:
    if model.level == "base_rate":
        return np.full(len(X), model.estimator)
    if model.level == "rule":
        flagged = X["n_offering_12m"].to_numpy() > 0
        return np.where(flagged, model.estimator["flagged"], model.estimator["not"])
    return model.estimator.predict_proba(X.to_numpy())[:, 1]


def predict(model: DilutionModel, rows_in: pd.DataFrame) -> pd.DataFrame:
    """Chance and band (1-5) for every row with the model's eligibility flag true."""
    missing = [c for c in model.features if c not in rows_in.columns]
    if missing:
        raise KeyError(f"missing feature columns: {missing}")
    r = rows_in[rows_in[model.eligibility].fillna(False).astype(bool)]
    p = _raw_chance(model, r[model.features].astype(float))
    if model.calibrator is not None:
        p = calibrate(model.calibrator, p)
    band = np.searchsorted(model.band_edges, p, side="right") + 1
    return pd.DataFrame({"cik": r["cik"].to_numpy(), "as_of": r["as_of"].to_numpy(),
                         "chance": p, "band": band})


def freeze(level: str) -> DilutionModel:
    """Refit `level` for target year 2023 exactly as a fold: inner-train rows with windows ended
    before 2022, settings, calibration and band edges from the 2022 rows (research R8)."""
    f = fold(2023)
    if level == "base_rate":
        est, cal, feats, p_valid = y[f["train"]].mean(), None, ["n_offering_12m"], None
    elif level == "rule":
        tr = f["train"].to_numpy()
        est = {"flagged": y[tr & FLAG].mean(), "not": y[tr & ~FLAG].mean()}
        cal, feats, p_valid = None, ["n_offering_12m"], None
    else:
        fit = fit_logistic(f) if level == "logistic" else fit_boosting(f)
        est, cal, feats = fit["estimator"], fit["calibrator"], fit["features"]
        p_valid = calibrate(cal, fit["valid_raw"])
    edges = (list(np.quantile(p_valid, [0.2, 0.4, 0.6, 0.8])) if p_valid is not None
             else [0.5, 0.5, 0.5, 0.5])
    return DilutionModel(level, list(feats), ELIG, est, cal, edges, SEED, TAGS["git_commit"],
                         TAGS["export_date"], TAGS["data_hash"],
                         str(rows.loc[f["train"], "window_end"].max().date()))


FREEZE_LEVEL = CHOSEN
frozen = freeze(FREEZE_LEVEL)
again = freeze(FREEZE_LEVEL)

# Dry run on rows without answers: each company's newest row, and the 2023+ feature rows
latest = pd.read_parquet(eda.EXPORT / "latest.parquet")
snap = pq.read_table(eda.EXPORT / "feature_snapshot.parquet",
                     filters=[("as_of", ">", eda.DEV_END)]).to_pandas()
assert ANSWER not in snap.columns and ANSWER not in latest.columns
for name, frame in {"latest": latest, "2023+ feature rows": snap}.items():
    out, out2 = predict(frozen, frame), predict(again, frame)
    eligible = int(frame[ELIG].fillna(False).astype(bool).sum())
    assert len(out) == eligible and out["chance"].between(0, 1).all() and out["chance"].notna().all()
    assert np.array_equal(out["chance"].to_numpy(), out2["chance"].to_numpy())
    print(f"{name}: {len(frame):,} rows, {eligible:,} eligible, all scored; "
          f"two runs identical: True; bands: {out['band'].value_counts().sort_index().to_dict()}")

with tempfile.TemporaryDirectory() as tmp, mlflow.start_run(run_name=f"frozen-{FREEZE_LEVEL}"):
    path = Path(tmp) / "dilution_model.joblib"
    joblib.dump(frozen, path)
    mlflow.set_tags({**TAGS, "level": FREEZE_LEVEL, "frozen": "candidate"})
    mlflow.log_params({"level": FREEZE_LEVEL, "trained_through": frozen.trained_through,
                       "n_features": len(frozen.features)})
    mlflow.log_artifact(str(path))
print(f"frozen candidate saved in MLflow: {FREEZE_LEVEL}, trained through "
      f"{frozen.trained_through}, band edges {np.round(frozen.band_edges, 4).tolist()}")

### Findings

*Export of 30 October 2026 (data hash `4bd17414f49b`), 409,576 learnable rows, 95 features
(`cum_split_after` left out: it holds future split ratios). All numbers are aggregates.*

**In one paragraph.** Dilution is well predictable. Gradient-boosted trees rank diluting
companies far better than the offering rule (AUC 0.87–0.90 against 0.71–0.74) and give
calibrated chances: the 20% of companies in the top band dilute 36–47% of the time, those in the
bottom band under 0.6%. The trees win in every test year on every measure, so they are chosen.

**The comparison (section 7)**

| level | log loss (mean ± SE) | AUC per test year | Brier per test year | top-tenth lift |
|---|---|---|---|---|
| base rate | 0.343 ± 0.027 | 0.50 | 0.075–0.120 | – |
| offering rule | 0.302 ± 0.021 | 0.71–0.74 | 0.072–0.108 | 1.8–2.2 |
| logistic regression | 0.250 ± 0.013 | 0.85–0.89 | 0.064–0.083 | 4.2–4.8 |
| **gradient boosting** | **0.229 ± 0.013** | **0.87–0.90** | **0.059–0.077** | **4.8–5.2** |

- **Choice:** the simplest level within one standard error of the best is the best itself
  (logistic 0.250 > 0.229 + 0.013), so **gradient boosting**. It beats the rule (log loss and
  AUC) and the base rate's Brier score in **every** test year (SC-004).
- The base rate's Brier score differs by year (0.075–0.120) because the dilution rate does
  (8–14%; 2020 was a year of share sales). 2022 covers January–June only.

**Bands (section 8).** Fixed edges from the year before each test year. In every test year the
real rate rises from band 1 to band 5 (SC-005), and forecasts sit close to the real rates:

| band | forecast (range over years) | real dilution rate |
|---|---|---|
| 1 | 0.4–0.6% | 0.3–0.6% |
| 2 | 1.0–1.5% | 0.7–1.4% |
| 3 | 2.2–4.1% | 2.0–3.5% |
| 4 | 7–13% | 7.7–13.5% |
| 5 | 38–48% | 36–47% |

Bands 4 and 5 over-forecast a little in 2021 (11% vs 7.7%; 48% vs 36%), the year after the
2020 wave of share sales.

**What the model uses (section 9).** Size and trading (traded value, market value, shares
outstanding: small, thinly traded companies dilute more), offerings and shelf registrations in the
last 3 and 12 months, unregistered sales, months since the last report, share growth, current
ratio and cash runway, loss-making. This matches the EDA. No single feature is near-perfect
(no leak sign).

**Family check (D60, five seeds).** Removing the **insider** family changes log loss by
−0.0005 to +0.0004 (within seed noise): insider activity adds nothing to this question.
Removing the **macro** family *improves* log loss in three of four years (by up to 0.0075 in
2021): macro features let the trees learn year-specific moods that do not carry over.

**Holdout-ready (section 10).** The frozen candidate (trained on windows to 29 December 2022,
calibration and band edges from 2022) scored all 6,406 eligible newest rows and all 240,212
eligible 2023+ feature rows; two independent fits gave identical predictions; saved in MLflow
(experiment `007-dilution`, run `frozen-boosting`).

**For the researcher to decide**

1. **Level 3 estimator:** keep scikit-learn's boosting or switch to LightGBM (D60; needs
   `brew install libomp` on the teammate's Mac). Expected to change little.
2. **Drop the macro family** from the trees (the family check says it hurts) and probably the
   insider family (it adds nothing): a simpler model, to be confirmed with a re-run.
3. **Logistic regression:** the smallest C tried (0.01) won in three folds; a smaller C might help
   it, though it is unlikely to close a 0.02 log-loss gap.
4. **The 46 split-like answers** could not be removed here (the check needs `sentinel.duckdb`).
5. **Next:** after approval, move `DilutionModel`, `freeze` and `predict` into
   `src/sentinel/models/dilution.py` with tests (T024–T025).